get all relevant sensititre and vitek files

In [ ]:
import pandas as pd
import os

df_vitek = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/VITEK_combined_02_26_interpreted_bin.csv"
)
df_sensititre = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/bin/Sensititre_combined_02_26_interpreted_bin.csv"
)

available_ids_vitek = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/VITEK/gff_card_401")
    if f.endswith(".gff")
}
available_ids_sensititre = {
    os.path.splitext(f)[0]
    for f in os.listdir("/groups/ds/Win-KID/BVBRC/sensititre/gff_card_401")
    if f.endswith(".gff")
}

# Remove unavialible IDs
df_vitek = df_vitek[
    df_vitek["Sample_ID_IfH"].astype(str).isin(available_ids_vitek)
].reset_index(drop=True)
df_sensititre = df_sensititre[
    df_sensititre["Sample_ID_IfH"].astype(str).isin(available_ids_sensititre)
].reset_index(drop=True)

# Remove non ECO samples
df_vitek = df_vitek[df_vitek["Organism_Code"] == "ECO"]
df_sensititre = df_sensititre[df_sensititre["Organism_Code"] == "ECO"]

df_vitek = df_vitek.dropna(subset=df_vitek.columns[2:], how="all")
df_sensititre = df_sensititre.dropna(subset=df_sensititre.columns[2:], how="all")

df_vitek.to_csv("/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/vitek.csv", index=False)
df_sensititre.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/sensititre.csv", index=False
)

print(len(df_vitek))
print(df_vitek.notna().sum(axis=1).mean())
print(len(df_sensititre))
print(df_sensititre.notna().sum(axis=1).mean())

1342
10.863636363636363
221
14.53393665158371


Cross validate EKP NA values
Random select EKP Phoenix values 
Random select EKP VITEK values CV

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Clean up vitek
df_vitek = pd.read_csv("/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/vitek.csv")
df_vitek_cleanup = pd.read_csv("samples_used_vitek_10.csv")
df_vitek_cleaned = df_vitek[
    df_vitek["Sample_ID_IfH"].isin(df_vitek_cleanup["Sample_ID_IfH"])
]

df_vitek_cleaned.to_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/vitek_cleaned.csv", index=False
)

# Clean up sensititre
df_sensititre = pd.read_csv("/groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/sensititre.csv")
df_sensisitre_cleanup = pd.read_csv("samples_used_sensititre_10.csv")
df_sensititre_cleaned = df_sensititre[
    df_sensititre["Sample_ID_IfH"].isin(df_sensisitre_cleanup["Sample_ID_IfH"])
]

df_sensititre_cleaned.to_csv("/sensititre_cleaned.csv", index=False)

Create and save splits

In [ ]:
import pandas as pd
from sklearn.model_selection import KFold
from pathlib import Path

N_REPEATS = 1
N_SPLITS = 4
OUTPUT_DIR = Path("")

df_vitek = pd.read_csv("vitek_cleaned.csv")
df_sensititre = pd.read_csv("sensititre_cleaned.csv")

remainder = len(df_sensititre) % N_SPLITS
if remainder > 0:
    # zufällig Zeilen auswählen, die entfernt werden
    df_sensititre = df_sensititre.sample(
        n=len(df_sensititre) - remainder, random_state=42
    ).reset_index(drop=True)


for repeat in range(N_REPEATS):
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=repeat)

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(df_sensititre)):
        seed = repeat * 100 + fold_idx
        run_name = f"repeat{repeat}_fold{fold_idx}"

        run_dir = OUTPUT_DIR / run_name
        run_dir.mkdir(parents=True, exist_ok=True)

        # -----------------------------
        # NA SPLIT
        # -----------------------------
        na_train = df_sensititre.iloc[test_idx]  # Change test and train
        na_test = df_sensititre.iloc[train_idx]  # Change test and train

        # -----------------------------
        # SAVE
        # -----------------------------
        na_train.to_csv(run_dir / "sensititre_train.csv", index=False)
        na_test.to_csv(run_dir / "sensititre_test.csv", index=False)

        print(f"Saved: {run_dir}")

Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold0
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold1
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold2
Saved: /groups/ds/Win-KID/BVBRC/BVBRC_devices/ECO/CV/repeat0_fold3


Generate DataSet counts

In [1]:
import pandas as pd

df_vitek = pd.read_csv("vitek_cleaned.csv")
df_sensititre = pd.read_csv("sensititre_cleaned.csv")


def count_s_r_per_antibiotic(df: pd.DataFrame) -> pd.DataFrame:
    # Identify antibiotic columns
    antibiotic_cols = [
        col for col in df.columns if col not in ["Sample_ID_IfH", "Organism_Code"]
    ]

    # Count S and R for each antibiotic
    sr_counts = pd.DataFrame(
        {
            "S": (df[antibiotic_cols] == "S").sum(),
            "R": (df[antibiotic_cols] == "R").sum(),
        }
    )

    # Calculate total number of available results
    sr_counts["Total"] = sr_counts["S"] + sr_counts["R"]

    # Remove antibiotics without any S or R result
    sr_counts = sr_counts[sr_counts["Total"] > 0]

    # Convert antibiotic names from index to column
    sr_counts = sr_counts.reset_index()
    sr_counts = sr_counts.rename(columns={"index": "Antibiotic"})

    return sr_counts


vitek_count = count_s_r_per_antibiotic(df_vitek)
print("VITEK")
print(vitek_count)
vitek_count.to_csv("vitek_count.csv", index=False)

df_sensititre_count = count_s_r_per_antibiotic(df_sensititre)
print("sensititre")
print(df_sensititre_count)
df_sensititre_count.to_csv("sensisitre_count.csv", index=False)

VITEK
                       Antibiotic     S    R  Total
0                       meropenem  1237    0   1237
1                       ertapenem   131   14    145
2                      tobramycin   138   40    178
3                     tigecycline     0    8      8
4         piperacillin-tazobactam  1032  103   1135
5                      cefotaxime   689  318   1007
6     amoxicillin-clavulanic acid   839  295   1134
7                      ampicillin    32  199    231
8                      gentamicin  1043  185   1228
9                        amikacin  1178   46   1224
10                    ceftriaxone    83  136    219
11                       cefepime   148   68    216
12                   trimethoprim    25   59     84
13  trimethoprim-sulfamethoxazole     0  307    307
14                    norfloxacin    71   13     84
15                  ciprofloxacin   686  536   1222
16                 nitrofurantoin   167    0    167
17                    ceftazidime  1004  210   1214
18    